# Lab 1: 실제 하드웨어를 위한 양자 회로 만들기

## Qiskit Global Summer School 2026

Lab 1에 오신 걸 환영합니다. 이 노트북에서는 양자 회로를 만들고, 실제 양자 하드웨어에서 돌릴 수 있게 다듬는 법을 배웁니다.

**배울 내용:**

1. [**Part 1: 기본 게이트와 양자 개념**](#part1): X, H, CX 게이트로 중첩과 얽힘을 만드는 법
2. [**Part 2: 회로 깊이**](#part2): 깊이가 왜 중요한지, GHZ 상태, 대칭성으로 깊이 줄이기
3. [**Part 3: 트랜스파일**](#part3): 실제 프로세서(133큐비트 IBM Heron 칩)의 제약에 회로 맞추기
4. [**Part 4: 노이즈 시뮬레이터에서 실행**](#part4): 노이즈 시뮬레이터에서 회로를 돌리고 결과 비교하기

각 파트는 앞의 내용 위에 쌓입니다. 끝에 가면 실제 하드웨어를 위한 효율적인 64큐비트 얽힘 상태를 만들게 되고, 이 기법은 Lab 2로 그대로 이어집니다.

## 시작하기 전에

> **이 랩은 처음부터 끝까지 로컬 *시뮬레이터*(`FakeTorino`)에서 돌아갑니다.** 여기서는 IBM Quantum 런타임 시간을 **전혀** 쓰지 않습니다. 모든 회로가 로컬에서 트랜스파일되고 실행됩니다. (Part 4 끝에서 실제 백엔드로 바꾸는 법을 보여줍니다.) 다만 답을 grader에 *제출*하려면 설정된 IBM Quantum 계정이 있어야 합니다. grade 셀에서 인증 오류가 나면 임포트 바로 아래의 계정 설정 셀을 보세요.

<details>
<summary><b>자주 겪는 설정 문제 (클릭해서 펼치기)</b></summary>

- **그림이 안 보이나요?** **VS Code**에서 플롯이 그림 대신 텍스트로 찍히면 셀을 다시 실행하세요. 이 노트북은 `display(...)`를 직접 호출하므로 Jupyter, JupyterLab, VS Code, Colab, qBraid 어디서든 그림이 나옵니다.
- **`jupyter notebook`을 못 찾나요?** 클래식 Notebook 대신 **JupyterLab**이 깔려 있을 수 있습니다. `jupyter lab`으로 실행하세요.
- **토폴로지/게이트 맵 플롯에서 Graphviz 오류가 나나요?** Graphviz를 설치하세요(`conda install -c conda-forge python-graphviz`, 또는 OS 패키지 관리자). 이 랩의 토폴로지 플롯은 `networkx` + `matplotlib`를 쓰므로 대부분은 필요 없습니다.
- **`pip`가 엉뚱한 환경에 설치하나요?** `!pip` 대신 `%pip` 매직을 쓰세요(아래 설치 셀처럼). `%pip`는 늘 실행 중인 커널에 설치합니다.

</details>

**참고할 Qiskit 문서:** [Circuits & gates](https://quantum.cloud.ibm.com/docs/api/qiskit/circuit) · [`QuantumCircuit`](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit) · [Transpilation](https://docs.quantum.ibm.com/guides/transpile) · [`Statevector`](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.quantum_info.Statevector)

## 필요한 라이브러리 설치 및 불러오기

In [ ]:
# 이 패키지들이 환경에 아직 없다면 이 셀을 실행하세요.
# (%pip 매직을 쓰므로 패키지가 항상 *실행 중인* 커널에 설치됩니다 —
#  일반 !pip가 엉뚱한 환경을 대상으로 할 수 있는 qBraid/Colab에서 중요합니다.)
%pip install 'qiskit[visualization]' qiskit-ibm-runtime qiskit-aer numpy networkx
%pip install --upgrade qc-grader

In [ ]:
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import matplotlib.pyplot as plt
import numpy as np

print("Imports loaded successfully — you're ready to go.")

In [ ]:
from qc_grader.challenges.qgss_2026 import (
    grade_lab1_ex1,
    grade_lab1_ex2,
    grade_lab1_ex3,
    grade_lab1_ex4,
    grade_lab1_ex5,
    grade_lab1_ex6,
    grade_lab1_ex7,
)

In [ ]:
# ── (선택) IBM Quantum 계정 설정 ──────────────────────────────────────
# grader는 답을 로컬에서 확인하지만, 답을 *제출*하려면 설정된 IBM Quantum
# 계정이 필요합니다. 새 Colab/qBraid 세션은 Lab 0에서 저장한 계정을
# 물려받지 않으므로, 아래 grade 셀이 인증 오류로 실패하면 Lab 0에서 썼던
# 것과 동일한 save_account(...) 호출을 실행하세요 — 예:
#
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<YOUR_IBM_QUANTUM_API_TOKEN>",
#     overwrite=True,
# )
#
# 환경마다 한 번만 하면 됩니다.

---

## Part 1: 기본 게이트와 양자 개념 <a id="part1"></a>

양자 계산은 고전 비트의 양자 버전인 **큐비트**를 사용해서 이뤄집니다. 큐비트는 $|0\rangle$ 상태에서 출발하고, **양자 게이트**로 이를 변환합니다. 가장 중요한 단일 큐비트 게이트와 2큐비트 게이트 세 개를 만나 봅시다.

### X 게이트 (비트 반전)

X 게이트는 큐비트 상태를 뒤집습니다. $|0\rangle \to |1\rangle$, $|1\rangle \to |0\rangle$. 고전 NOT 게이트의 양자 버전입니다.

기하학적으로는 블로흐 구의 X축을 중심으로 한 $\pi$(180°) 회전입니다. 행렬은 다음과 같습니다.

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$$

> **아래 출력 읽는 법:** 각 상태를 두 방식으로 보여줍니다. 하나는 상태 *벡터*(LaTeX 형태로, 각 항은 $|0\rangle$ 같은 기저 상태에 곱해진 진폭)이고, 다른 하나는 **블로흐 구**(단일 큐비트의 기하학적 그림)입니다. 기저 상태가 측정될 확률은 그 진폭 크기의 제곱입니다.

In [ ]:
# 단일 큐비트에 X 게이트를 적용하는 회로 만들기
qc = QuantumCircuit(1)
qc.x(0)
display(qc.draw("mpl"))

# 결과 상태 살펴보기: 벡터(LaTeX)로, 그리고 블로흐 구로
sv = Statevector(qc)
display(sv.draw("latex"))
display(plot_bloch_multivector(sv))

### H 게이트 (중첩)

하다마드 게이트는 큐비트를 **중첩**에 놓습니다. $|0\rangle$이면서 동시에 $|1\rangle$인, 두 상태에 걸쳐 있는 상태죠.

$$H|0\rangle = \frac{1}{\sqrt{2}}\bigl(|0\rangle + |1\rangle\bigr)$$

이 상태의 큐비트를 측정하면 $|0\rangle$과 $|1\rangle$이 각각 50% 확률로 나옵니다. X처럼 H도 회전인데, 이번에는 블로흐 구에서 X축과 Z축 사이 한가운데 축을 중심으로 한 $\pi$ 회전입니다. 행렬은 다음과 같습니다.

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
display(qc.draw("mpl"))

sv = Statevector(qc)
display(sv.draw("latex"))
display(plot_bloch_multivector(sv))

### CX (CNOT) 게이트와 얽힘

CX 게이트는 **2큐비트** 게이트입니다. **제어** 큐비트가 $|1\rangle$일 때만 **대상** 큐비트를 뒤집습니다. 제어가 중첩에 있으면 흥미로운 일이 벌어집니다. 두 큐비트가 **얽힙니다**. 두 큐비트의 측정 결과가 고전적으로는 설명할 수 없을 만큼 완벽하게 맞물립니다.

첫 **벨 상태**를 만들며 직접 봅시다.

In [ ]:
# 벨 상태 (|00⟩ + |11⟩) / √2 만들기
qc = QuantumCircuit(2)
qc.h(0)  # 큐비트 0을 중첩에 놓기
qc.cx(0, 1)  # 큐비트 1을 큐비트 0과 얽히게 하기
display(qc.draw("mpl"))

sv = Statevector(qc)
display(sv.draw("latex"))

#### 벨 상태 이해하기

방금 만든 상태는 $\frac{1}{\sqrt{2}}\bigl(|00\rangle + |11\rangle\bigr)$입니다. 무슨 뜻일까요?

- 두 큐비트를 측정하면 $|00\rangle$이나 $|11\rangle$이 각각 50% 확률로 나옵니다.
- $|01\rangle$이나 $|10\rangle$은 **절대** 안 나옵니다. 두 큐비트가 완벽하게 맞물려 있습니다.
- 이 상관은 두 큐비트가 아무리 멀리 떨어져도 유지됩니다. 이것이 **얽힘**이고, 양자 컴퓨팅을 강력하게 만드는 핵심 자원입니다.

### 연습 1: $|01\rangle + |10\rangle$ 벨 상태 만들기

방금 $\frac{1}{\sqrt{2}}\bigl(|00\rangle + |11\rangle\bigr)$를 만들었습니다. 이제 여러분 차례입니다.

**과제:** 벨 상태 $\frac{1}{\sqrt{2}}\bigl(|01\rangle + |10\rangle\bigr)$를 준비하는 회로를 만드세요.

<details>
<summary><b>힌트</b></summary>

$|00\rangle + |11\rangle$ 만드는 법은 이미 압니다. 큐비트 하나를 뒤집으려면 어떤 게이트를 더하면 될까요?
</details>

문서: [`QuantumCircuit` gates](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) (`.x()`, `.h()`, `.cx()` 참고)

In [ ]:
qc = QuantumCircuit(2)

## 여기에 코드를 작성하세요 ##


## 답 확인
display(qc.draw("mpl"))
sv = Statevector(qc)
display(sv.draw("latex"))

In [ ]:
grade_lab1_ex1(qc)

#### 같은 상태로 가는 여러 길

양자 회로의 재미있는 점 하나는, 같은 상태를 여러 방식으로 준비할 수 있다는 것입니다. $|01\rangle + |10\rangle$를 만드는 방법만 해도 (적어도) 세 가지입니다.

<details>
<summary><b>클릭해서 세 가지 풀이 보기</b></summary>

**방법 1:** 벨 회로 *앞에* X 걸기

```python
qc = QuantumCircuit(2)
qc.x(1)        # 먼저 큐비트 1을 |1⟩로 뒤집기
qc.h(0)
qc.cx(0, 1)
```

**방법 2:** 벨 회로 *뒤에* 큐비트 1에 X 걸기

```python
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.x(1)        # 얽힌 뒤 큐비트 1 뒤집기
```

**방법 3:** 벨 회로 *뒤에* 큐비트 0에 X 걸기

```python
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.x(0)        # 얽힌 뒤 큐비트 0 뒤집기
```

셋 다 같은 상태를 만듭니다. 큐비트 1이 아니라 큐비트 0을 뒤집는 방법 3은 특히 뜻밖일 수 있습니다. 하지만 두 큐비트가 얽혀 있어서, 어느 쪽을 뒤집든 $|00\rangle \leftrightarrow |01\rangle$과 $|11\rangle \leftrightarrow |10\rangle$이 맞바뀌어 결국 $|01\rangle + |10\rangle$이 됩니다.

같은 상태에 서로 다른 회로로 도달할 수 있다는 이 생각은, 실제 하드웨어에 맞춰 최적화를 시작할 때 중요해집니다.
</details>

### 선택: 상대 위상

지금까지 $|00\rangle + |11\rangle$과 $|01\rangle + |10\rangle$을 봤습니다. 그런데 벨 상태는 사실 **네 개**입니다. 나머지 둘에는 **마이너스 부호**, 곧 상대 위상이 들어갑니다.

$$\frac{1}{\sqrt{2}}\bigl(|00\rangle - |11\rangle\bigr) \qquad \text{그리고} \qquad \frac{1}{\sqrt{2}}\bigl(|01\rangle - |10\rangle\bigr)$$

이 위상을 넣는 게 **Z 게이트**입니다. $|0\rangle$은 그대로 두고 $|1\rangle \to -|1\rangle$로 보냅니다.

$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

마이너스 부호는 표준 기저에서 측정 확률을 바꾸지 않습니다(여전히 50/50입니다). 하지만 물리적으로 실재하고, 이후 계산에서 상태가 간섭하는 방식에 영향을 줍니다.

In [ ]:
# Z 게이트를 더해 |00⟩ - |11⟩ 만들기
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.z(0)  # qc.z(1)로 바꿔 보세요 -- 결과는 같습니다!
display(qc.draw("mpl"))

sv = Statevector(qc)
display(sv.draw("latex"))

**선택 심화 연습:** $\frac{1}{\sqrt{2}}\bigl(|01\rangle - |10\rangle\bigr)$을 만들 수 있나요? X 게이트와 Z 게이트에서 배운 걸 조합해 보세요. (풀이는 없습니다. 직접 실험해 보세요.)

In [ ]:
qc = QuantumCircuit(2)

## 여기에 코드를 작성하세요 ##


## 확인
display(qc.draw("mpl"))
sv = Statevector(qc)
display(sv.draw("latex"))

### Part 1 요약

**이번에 익힌 것:**
- **X 게이트**: $|0\rangle \leftrightarrow |1\rangle$을 뒤집음 (비트 반전 / X축 $\pi$ 회전)
- **H 게이트**: 중첩 $|0\rangle \to \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$을 만듦
- **CX 게이트**: 두 큐비트를 얽히게 함, 양자 우위의 열쇠
- 같은 양자 상태를 **여러 다른 회로**로 준비할 수 있음
- *선택*: 상대 위상($\pm$ 부호)은 물리적으로 의미가 있음

---

## Part 2: 회로 깊이 <a id="part2"></a>

실제 양자 하드웨어에서는 게이트마다 오류가 조금씩 딸려 옵니다. 게이트 층이 많은 회로일수록 노이즈가 더 쌓이고 결과가 나빠집니다. 이 층의 개수를 **회로 깊이**라 하고, 이를 줄이는 건 양자 컴퓨팅에서 손꼽히게 중요한 실전 기술입니다.

이 파트에서는 깊이가 무엇인지 배우고, 더 큰 얽힘 상태를 만들고, **대칭성**으로 회로 깊이를 크게 줄이는 법을 봅니다.

### GHZ 상태: 규모를 키운 얽힘

벨 상태는 큐비트 2개를 얽힙니다. **GHZ 상태**(Greenberger–Horne–Zeilinger)는 이걸 큐비트 $N$개로 넓힙니다.

$$|\text{GHZ}_N\rangle = \frac{1}{\sqrt{2}}\bigl(|00\ldots0\rangle + |11\ldots1\rangle\bigr)$$

큐비트 3개라면 $\frac{1}{\sqrt{2}}\bigl(|000\rangle + |111\rangle\bigr)$입니다. 측정하면 모든 큐비트가 같은 값입니다. 전부 0이거나 전부 1이죠.

In [ ]:
# 3큐비트 GHZ 상태
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
display(qc.draw("mpl"))

sv = Statevector(qc)
display(sv.draw("latex"))

### N큐비트 GHZ 상태를 만드는 두 가지 방법

GHZ 상태를 만드는 자연스러운 방법은 (적어도) 두 가지입니다.

1. **큐비트 0에서 팬아웃**: 큐비트 0에 H를 걸고, 큐비트 0에서 나머지 모든 큐비트로 CX를 겁니다. CX(0,1), CX(0,2), …, CX(0,N−1).
2. **이웃을 따라 체인**: 큐비트 0에 H를 걸고, 얽힘을 체인을 따라 넘깁니다. CX(0,1), CX(1,2), …, CX(N−2,N−1).

둘 다 똑같은 GHZ 상태를 만듭니다. 다시 쓸 수 있게 함수로 구현해 봅시다.

In [ ]:
def ghz_fan(n):
    """팬아웃 방식 GHZ: 모든 CX 게이트가 큐비트 0을 제어로 사용."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(1, n):
        qc.cx(0, i)
    return qc


def ghz_chain(n):
    """체인 방식 GHZ: 각 CX가 얽힘을 다음 큐비트로 넘김."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    return qc


print("Fan-out (5 qubits):")
display(ghz_fan(5).draw("mpl"))
print("\nChain (5 qubits):")
display(ghz_chain(5).draw("mpl"))

### 회로 깊이란?

양자 회로의 **깊이**는, 병렬로 돌릴 수 있는 게이트를 최대한 병렬로 돌린다고 할 때 회로를 실행하는 데 필요한 시간 단계(층)의 개수입니다.

**병렬 실행 규칙:**
- 두 게이트는 **완전히 다른 큐비트**에 작용할 때만 **같은 층**에서 돌 수 있습니다.
- 두 게이트가 큐비트를 하나라도 공유하면 **다른 층**에서 돌아야 합니다.

> **주의:** 회로 그림은 착각을 줄 수 있습니다. 그림에서 게이트가 나란히 놓여 있어도, 큐비트를 공유하면 순차적으로 돌아갑니다. 깊이가 중요할 때는 늘 코드로 확인하세요.

In [ ]:
# 예시 1: 독립적인 X 게이트 두 개 → 깊이 1 (병렬로 실행됨)
qc1 = QuantumCircuit(2)
qc1.x(0)
qc1.x(1)
print("Two independent X gates -- depth:", qc1.depth())
display(qc1.draw("mpl"))

# 예시 2: 큐비트 0을 공유하는 CX 게이트 두 개 → 깊이 2 (순차적이어야 함)
qc2 = QuantumCircuit(3)
qc2.cx(0, 1)
qc2.cx(0, 2)
print("\nCX(0,1) then CX(0,2) -- depth:", qc2.depth())
display(qc2.draw("mpl"))

# 예시 3: 서로 겹치지 않는 큐비트의 CX 게이트 두 개 → 깊이 1 (병렬)
qc3 = QuantumCircuit(4)
qc3.cx(0, 1)
qc3.cx(2, 3)
print("\nCX(0,1) and CX(2,3) in parallel -- depth:", qc3.depth())
display(qc3.draw("mpl"))

### 연습 2: 회로 깊이 예측하기

아래 회로들을 보고, 확인 셀을 실행하기 **전에** 깊이를 **예측**해 보세요. 어떤 게이트가 병렬로 돌 수 있는지 따져 보세요.

다음 셀에서 표시된 두 줄에 `your_answer_a`와 `your_answer_b`를 예측한 정수로 넣은 뒤, 확인 셀과 grader를 실행하세요.

<details>
<summary><b>힌트</b></summary>

게이트마다 어떤 큐비트를 건드리는지 그려 보고 층을 나눠 보세요. 두 게이트는 완전히 다른 큐비트에 작용할 때만 한 층을 같이 쓸 수 있습니다.
</details>

In [ ]:
# 회로 A
qc_a = QuantumCircuit(4)
qc_a.h(0)
qc_a.h(1)
qc_a.cx(0, 2)
qc_a.cx(1, 3)
qc_a.cx(2, 3)
display(qc_a.draw("mpl"))

# 회로 A의 깊이 예측값을 your_answer_a에 정수로 설정하세요:
## 여기에 코드를 작성하세요 ##


# 회로 B
qc_b = QuantumCircuit(3)
qc_b.h(0)
qc_b.cx(0, 1)
qc_b.h(2)
qc_b.cx(2, 1)
display(qc_b.draw("mpl"))

# 회로 B의 깊이 예측값을 your_answer_b에 정수로 설정하세요:
## 여기에 코드를 작성하세요 ##

# 위 두 예측을 설정한 뒤, 다음 셀을 실행해 확인하고 grader를 실행하세요.

In [ ]:
# 답 확인하기!
print("Circuit A depth:", qc_a.depth())
print("Circuit B depth:", qc_b.depth())

In [ ]:
grade_lab1_ex2({"a": your_answer_a, "b": your_answer_b})

<details>
<summary><b>클릭해서 설명 보기</b></summary>

**회로 A (깊이 3):**
- 층 1: H(0)과 H(1) (다른 큐비트, 병렬)
- 층 2: CX(0,2)와 CX(1,3) (다른 큐비트, 병렬)
- 층 3: CX(2,3)

**회로 B (깊이 3):**
- 층 1: H(0)과 H(2) (다른 큐비트, 병렬)
- 층 2: CX(0,1) (H(0)을 기다려야 함)
- 층 3: CX(2,1) (앞의 CX와 큐비트 1을 공유하므로 순차적이어야 함)

회로 B의 그림만 보면 CX(0,1)과 CX(2,1)이 병렬일 것 같지만, 둘 다 큐비트 1을 씁니다.
</details>

### GHZ 회로의 깊이

앞서 만든 두 GHZ 구성의 깊이를 확인해 봅시다. 한쪽이 나을 것 같지만, 과연 그럴까요…

In [ ]:
for n in [5, 10, 20]:
    fan_depth = ghz_fan(n).depth()
    chain_depth = ghz_chain(n).depth()
    print(f"n = {n:2d}:  fan depth = {fan_depth},  chain depth = {chain_depth}")

두 회로는 **깊이가 같습니다**. 둘 다 $N$이죠(H 게이트에 1 + CX 게이트에 $N-1$). 왜 그럴까요?

- **팬아웃**: 모든 CX가 큐비트 0을 제어로 공유하니, 어느 것도 병렬로 못 돕니다.
- **체인**: 각 CX가 앞 CX의 결과에 기댑니다. 큐비트 $i$가 얽혀야 큐비트 $i+1$을 얽힐 수 있으니까요.

두 구성 다 큐비트 수에 대해 *선형*입니다. 큐비트 100개면 게이트 100층인데, 노이즈 있는 하드웨어엔 너무 많습니다. 더 잘할 수 있을까요?

### 깊이 줄이기: 가운데에서 시작하기

핵심은 이겁니다. GHZ 상태는 **대칭적**이라 모든 큐비트가 같은 중첩에 놓입니다. 굳이 한쪽 끝에서 시작할 이유가 없습니다.

H 게이트를 **가운데** 큐비트에 놓고 얽힘을 양쪽으로 동시에 키우면(왼쪽 체인 하나, 오른쪽 체인 하나가 나란히 진행), 두 절반이 함께 나아갑니다. 이렇게 하면 CX 깊이가 대략 **절반**으로 줄어듭니다.

In [ ]:
def ghz_half_depth(n, add_barriers=False):
    """가운데에서 시작해 양쪽 방향으로 키워 나가는 GHZ.

    add_barriers=True를 넘기면 바깥으로 뻗는 각 단계 뒤에 배리어를 넣습니다.
    이러면 회로 그림에서 병렬 층이 눈에 보입니다. 배리어는 하나의 층으로
    세어지므로, 깊이는 항상 배리어가 없는 회로에서 측정하세요.
    """
    qc = QuantumCircuit(n)
    mid = n // 2
    qc.h(mid)

    # 왼쪽과 오른쪽으로 동시에 퍼뜨리기
    for step in range(1, n):
        left = mid - step
        right = mid + step
        if left >= 0:
            qc.cx(left + 1, left)  # 왼쪽으로 퍼뜨리기
        if right < n:
            qc.cx(right - 1, right)  # 오른쪽으로 퍼뜨리기
        if add_barriers:
            qc.barrier()
        if left <= 0 and right >= n - 1:
            break
    return qc


# 병렬 층이 잘 보이도록 배리어를 넣어 그리고...
display(ghz_half_depth(20, add_barriers=True).draw("mpl"))

# ...깊이는 배리어가 없는 회로에서 측정합니다:
qc_half = ghz_half_depth(20)
print(f"Half-depth GHZ (20 qubits): depth {qc_half.depth()}")
print(f"Linear GHZ (20 qubits):     depth {ghz_chain(20).depth()}")

### 한 걸음 더: 재귀적 팬아웃

절반으로 줄이는 요령은 아이디어를 딱 한 번만 썼습니다. 작업을 두 갈래로 나눈 거죠. 그런데 이걸 **재귀적으로** 쓸 수 있습니다. 가운데 큐비트가 왼쪽과 오른쪽 큐비트를 얽히고 나면, 새로 얽힌 그 큐비트들이 *각자* 얽힘을 더 멀리 퍼뜨리는 새 출발점이 됩니다.

각 층에서는, 아직 안 얽힌 이웃을 둔 얽힌 큐비트마다 CX를 하나씩 겁니다. 그러면 층마다 얽힌 큐비트 수가 **두 배**로 불어납니다.

| 층 | 동작 | 얽힌 큐비트 |
|-------|--------|-----------------|
| 0 | 큐비트 0에 H | 1 |
| 1 | CX(0,1) | 2 |
| 2 | CX(0,2) + CX(1,3) | 4 |
| 3 | CX(0,4) + CX(1,5) + CX(2,6) + CX(3,7) | 8 |
| 4 | CX(0,8) + CX(1,9) + … + CX(7,15) | 16 |

깊이가 $N$ 대신 $\lceil\log_2(N)\rceil + 1$로 늘어납니다. **지수적 개선**이죠.

$N = 16$이면 깊이는 $\log_2(16) + 1 = 5$입니다.

> **참고:** 위 표는 설명을 단순하게 하려고 재귀적 팬아웃을 가운데가 아니라 **큐비트 0**에서 시작하도록 인덱싱했습니다. 어디서 시작하든 $O(\log N)$ 깊이 스케일링은 그대로입니다. 층마다 얽힌 큐비트가 두 배가 되는 게 핵심이니까요. 아래 연습 3은 이 큐비트 0 인덱싱을 씁니다.

### 연습 3: 16큐비트 깊이 5 GHZ 상태 만들기

**과제:** 재귀적 팬아웃으로 깊이가 정확히 5인 16큐비트 GHZ 회로를 만드세요.

**전략:** CX 층마다 이미 얽힌 큐비트가 새 큐비트를 하나씩 얽혀야 합니다. CX 층 $k$개를 거치면 얽힌 큐비트가 $2^k$개가 됩니다.

**참고:** 지금은 하드웨어가 아닌 추상적인 상황이라, 어떤 큐비트든 다른 어떤 큐비트에도 연결할 수 있습니다. 하드웨어 제약은 Part 3에서 다룹니다.

In [ ]:
qc = QuantumCircuit(16)

# 층 0: 하다마드 -- 큐비트 1개를 중첩에
qc.h(0)

# 층 1: CX 게이트 1개 → 얽힌 큐비트 2개
## 여기에 코드를 작성하세요 ##


# 층 2: CX 게이트 2개 → 얽힌 큐비트 4개
## 여기에 코드를 작성하세요 ##


# 층 3: CX 게이트 4개 → 얽힌 큐비트 8개
## 여기에 코드를 작성하세요 ##


# 층 4: CX 게이트 8개 → 얽힌 큐비트 16개
## 여기에 코드를 작성하세요 ##


# 확인
display(qc.draw("mpl"))
print("Depth:", qc.depth())
assert qc.depth() == 5, f"Expected depth 5, got {qc.depth()}"

sv = Statevector(qc)
probs = np.abs(sv.data) ** 2
assert np.isclose(probs[0], 0.5) and np.isclose(probs[-1], 0.5), (
    "Not a valid GHZ state!"
)
print("Success — valid 16-qubit GHZ state with depth 5.")

In [ ]:
grade_lab1_ex3(qc)

### Part 2 요약

**이번에 익힌 것:**
- **회로 깊이**: 순차적 게이트 층의 개수 (낮을수록 노이즈가 적음)
- **GHZ 상태**: $N$큐비트 얽힘 $\frac{1}{\sqrt{2}}(|00\ldots0\rangle + |11\ldots1\rangle)$
- 선형 GHZ 회로(팬아웃이든 체인이든)의 깊이는 $N$
- **가운데에서 시작하면** 깊이가 절반
- **재귀적 팬아웃**은 깊이 $\lceil\log_2(N)\rceil + 1$, 지수적으로 더 나음

---

## Part 3: 트랜스파일 <a id="part3"></a>

지금까지는 추상 큐비트와 H, CX 같은 익숙한 게이트로 회로를 만들었습니다. 그런데 실제 양자 하드웨어에는 구체적인 제약이 있습니다.

1. **네이티브 게이트가 정해져 있음**: 프로세서는 정해진 게이트 집합만 지원합니다. IBM Heron 프로세서는 $\{R_Z, \sqrt{X}, X, CZ\}$를 씁니다. H도 CX도 없습니다.
2. **연결이 제한적임**: 모든 큐비트 쌍이 물리적으로 이어져 있진 않습니다. 안 이어진 큐비트 사이에 2큐비트 게이트가 필요하면 SWAP 게이트를 넣어야 합니다(하나당 CZ 게이트 3개 비용).
3. **물리 큐비트에 배정해야 함**: 추상 큐비트 0은 칩 위의 특정 물리 큐비트에 대응돼야 합니다.

**트랜스파일**은 추상 회로를 이 제약을 모두 지키는 회로로 바꾸는 과정입니다. 직접 봅시다.

In [ ]:
from qiskit import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeTorino

# FakeTorino는 133큐비트 IBM Heron 프로세서를 시뮬레이션합니다
backend = FakeTorino()
print(f"Backend: {backend.name}")
print(f"Qubits: {backend.num_qubits}")
print(
    f"Native gates: {sorted(g for g in backend.operation_names if g not in ['reset', 'delay', 'measure', 'id', 'if_else', 'for_loop', 'switch_case'])}"
)

작게 시작하겠습니다. **5큐비트** 체인 GHZ를 트랜스파일해 전후를 쉽게 비교합니다. (크기가 바뀐다고 당황하지 마세요. Part 2는 큐비트 20개까지 갔고, 연습 7에서 다시 64큐비트 GHZ로 올라갑니다.)

In [ ]:
# 간단한 5큐비트 체인 GHZ 회로 트랜스파일
qc = ghz_chain(5)
print("Original circuit:")
display(qc.draw("mpl"))

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

qc_with_meas = qc.copy()
qc_with_meas.measure_all()
isa_qc = pm.run(qc_with_meas)

print("\nTranspiled circuit:")
display(isa_qc.draw("mpl", idle_wires=False))

print(f"\nBefore: depth {qc.depth()}, gates {dict(qc.count_ops())}")
print(f"After:  depth {isa_qc.depth()}, gates {dict(isa_qc.count_ops())}")

#### 무엇이 바뀌었나?

트랜스파일된 회로를 보세요.

- **H가 사라지고** $R_Z$와 $\sqrt{X}$ 게이트 조합(네이티브 단일 큐비트 게이트)으로 바뀌었습니다
- **CX가 사라지고** 단일 큐비트 게이트로 둘러싸인 CZ 게이트로 바뀌었습니다
- **물리 큐비트 번호**가 나타납니다. 추상 큐비트가 특정 하드웨어 큐비트에 배정된 거죠
- **깊이가 늘었습니다**. 추상 게이트 하나가 네이티브 게이트 여러 개로 펼쳐지니까요

다행히 트랜스파일러가 큐비트 5개를 모두 체인으로 이어진 배치에 얹어서 **SWAP 게이트가 필요 없었습니다**. 늘 이렇지는 않습니다.

### 곁길: 토폴로지를 무시하면?

우리 체인 GHZ가 깔끔하게 트랜스파일된 건 큐비트 0-1-2-3-4가 Heron에서 물리적으로 이어져 있기 때문입니다. 그럼 **같은** 물리 배치에 **팬아웃** 패턴을 쓰면 어떻게 될까요? 큐비트 0은 큐비트 1과는 이어져 있지만 큐비트 2, 3, 4와는 **안 이어져** 있습니다. 트랜스파일러는 `CX(0, 2)`, `CX(0, 3)`, `CX(0, 4)`를 어떻게든 해내야 합니다.

그 해법이 **SWAP 게이트 넣기**입니다. 큐비트 0의 상태를 체인을 따라 밀어, 대상 큐비트 옆에 올 때까지 물리적으로 옮깁니다. SWAP 하나는 실제 하드웨어에서 **CZ 게이트 3개**로 분해됩니다. 트랜스파일이 회로를 *눈에 띄게* 다시 쓰는 장면이죠. 게이트 수와 깊이가 뛰는 걸 지켜보세요. 직접 봅시다.

> **측정에 관한 참고:** SWAP이 논리 큐비트를 다른 물리 배선으로 옮기면, 트랜스파일러는 고전 비트마다 여러분이 의도한 큐비트를 기록하도록 **측정 경로도 다시 잡아** 줍니다. 이 뒤처리는 알아서 되지만, 그래서 트랜스파일된 그림의 측정 부분이 원래 회로와 딱 맞아떨어지지 않을 수 있습니다.

In [ ]:
from qiskit import transpile

# 같은 팬아웃 GHZ, 같은 물리 배치 (Heron의 체인 0-1-2-3-4)
qc_fan = ghz_fan(5)
qc_fan.measure_all()

# SWAP을 보이게 유지한 채 트랜스파일 (아직 네이티브 게이트로 분해하지 않음)
isa_fan_swaps = transpile(
    qc_fan,
    coupling_map=backend.coupling_map,
    basis_gates=["h", "cx", "swap"],
    initial_layout=[0, 1, 2, 3, 4],
    optimization_level=1,
)

print("Fan-out GHZ on the line 0-1-2-3-4 (SWAPs kept visible):")
display(isa_fan_swaps.draw("mpl", idle_wires=False))
print(f"SWAPs inserted: {isa_fan_swaps.count_ops().get('swap', 0)}")
print(f"CX gates:       {isa_fan_swaps.count_ops().get('cx', 0)}")
print("(Each SWAP becomes 3 CZ gates in the final native circuit.)")

# 이제 네이티브 게이트로 완전히 분해 -- 실제로 하드웨어에서 도는 회로입니다:
isa_fan_native = pm.run(qc_fan)
print(
    f"\nFully transpiled to native gates: depth {isa_fan_native.depth()}, "
    f"CZ gates {isa_fan_native.count_ops().get('cz', 0)}"
)
display(isa_fan_native.draw("mpl", idle_wires=False, fold=-1))

### 브리지 게이트 항등식

SWAP만이 비국소 CX를 다루는 유일한 방법은 아닙니다. **브리지 게이트**라는 재치 있는 회로 항등식이 있어, 체인 `0 — 1 — 2`에서 최근접 이웃 게이트만으로 `CX(0, 2)`를 구현합니다. 상태를 **전혀 옮기지 않고서**요.

$$\text{CX}(0, 2) \;=\; \text{CX}(0, 1)\,\cdot\,\text{CX}(1, 2)\,\cdot\,\text{CX}(0, 1)\,\cdot\,\text{CX}(1, 2)$$

이 순서로 놓인 최근접 이웃 CX 네 개가 장거리 `CX(0, 2)` 하나와 **똑같은** 변환을 만듭니다. 가운데 큐비트가 다리 역할을 합니다. 그 상태가 잠깐 뒤집혔다 다시 뒤집혀 제자리로 오는 사이, 얽힘 작용이 큐비트 0에서 큐비트 2로 건너갑니다.

브리지는 중간 큐비트 하나를 거쳐 비국소 CX 하나가 필요할 때 쓸 만합니다. 경로가 더 복잡해지면 트랜스파일러는 대개 다시 SWAP을 고릅니다. 그래도 이 항등식은 비국소 게이트의 비용을 눈에 보이게 해 줍니다. 이 요령을 써도, "공짜"라던 비국소 CX가 국소 CX보다 CX 게이트를 **4배** 씁니다.

In [ ]:
# 브리지 항등식 확인: CX(0, 2) == CX(0,1) CX(1,2) CX(0,1) CX(1,2)

# 직접: 장거리 CX 하나 (얽힘 작용이 보이도록 큐비트 0에 H를 걸어 둠)
qc_direct = QuantumCircuit(3)
qc_direct.h(0)
qc_direct.cx(0, 2)

# 브리지: 최근접 이웃 CX 게이트 네 개
qc_bridge = QuantumCircuit(3)
qc_bridge.h(0)
qc_bridge.cx(0, 1)
qc_bridge.cx(1, 2)
qc_bridge.cx(0, 1)
qc_bridge.cx(1, 2)

sv_direct = Statevector(qc_direct)
sv_bridge = Statevector(qc_bridge)

print("Direct  CX(0, 2):", sv_direct)
print("Bridge form:    ", sv_bridge)
print("States equal?   ", sv_direct.equiv(sv_bridge))

### 연습 4: 브리지 항등식으로 최근접 이웃 GHZ 만들기

팬아웃 패턴으로 만든 3큐비트 GHZ가 있습니다.

```python
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 2)   # <-- 이 CX는 0-1-2 체인에서 비국소!
```

마지막 게이트 `CX(0, 2)`는 이웃끼리만 이어진 하드웨어에서는 바로 못 돕니다.

**과제:** 같은 3큐비트 GHZ 상태를 만들되, **브리지 항등식**으로 `CX(0, 2)`를 최근접 이웃 CX 네 개로 바꾸세요. 최종 회로는 `CX(0, 1)`과 `CX(1, 2)`만 써야 합니다. `CX(0, 2)`는 절대 안 됩니다.

다 하면, 체인 형태(`H(0); CX(0, 1); CX(1, 2)`)와 CX 개수를 비교해 보세요. 그 차이가, 하드웨어는 체인을 원하는데 팬아웃 모양을 고집할 때 치르는 실제 비용입니다.

In [ ]:
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)

## 여기에 코드를 작성하세요: 브리지 항등식으로 비국소 CX(0, 2)를 대체하세요 ##


## 확인
display(qc.draw("mpl"))
sv = Statevector(qc)
print("Your state:", sv)

# 3큐비트 GHZ 상태여야 함
probs = np.abs(sv.data) ** 2
assert np.isclose(probs[0], 0.5) and np.isclose(probs[-1], 0.5), (
    "Not a valid 3-qubit GHZ state!"
)

# 최근접 이웃 CX 게이트만 써야 함 (큐비트를 건너뛰지 않음)
non_local = []
for instr in qc.data:
    if instr.operation.name == "cx":
        q_indices = [qc.find_bit(q).index for q in instr.qubits]
        if abs(q_indices[0] - q_indices[1]) != 1:
            non_local.append(tuple(q_indices))
assert not non_local, f"Non-nearest-neighbor CX found: {non_local}"

print(f"Valid GHZ using only nearest-neighbor CX gates ({qc.count_ops().get('cx', 0)} CX total).")

In [ ]:
grade_lab1_ex4(qc)

<details>
<summary><b>클릭해서 풀이 보기</b></summary>

```python
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
# CX(0, 2)를 대신하는 브리지 항등식: 최근접 이웃 CX 네 개
qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(0, 1)
qc.cx(1, 2)
```

이대로도 되지만, 원래 `CX(0, 1)`과 브리지의 첫 `CX(0, 1)`이 붙어 있고 `CX(0, 1) · CX(0, 1) = I`(어떤 CX든 자기 자신이 역)라는 점을 눈여겨보세요. 그래서 이 둘을 지우면 다음이 남습니다.

```python
qc.h(0)
qc.cx(1, 2)
qc.cx(0, 1)
qc.cx(1, 2)
```

**CX 게이트 3개.** 체인 형태 `H(0); CX(0,1); CX(1,2)`는 **2개**입니다. 이렇게 정리해도, 최근접 이웃 칩에서 팬아웃 모양을 고집하면 비국소 홉마다 CX가 하나씩 더 듭니다. 체인이 길어지고 CX 거리가 멀어질수록 이 오버헤드는 금세 커집니다. 그래서 처음부터 토폴로지를 염두에 두고 회로를 짜는 것(Part 3의 나머지)이 진짜 이득입니다.
</details>

### 연습 5: 라인 위의 효율적인 5큐비트 GHZ

FakeTorino의 물리 큐비트 0–4는 체인을 이룹니다: 0-1-2-3-4. 이 연결을 지키면서 깊이를 최소화하는 GHZ 회로를 만들어 봅시다.

**과제:** 가운데 큐비트에서 시작해 추상 깊이가 **4**인 5큐비트 GHZ 회로를 만드세요.

<details>
<summary><b>힌트</b></summary>

Part 2의 "가운데에서 시작하기" 요령을 떠올리세요. 가운데 큐비트에 H를 놓고 양쪽 끝으로 키워 나가면 됩니다.
</details>

In [ ]:
qc = QuantumCircuit(5)

## 여기에 코드를 작성하세요: 가운데에서 시작하는 깊이 4 GHZ를 만드세요 ##


# 확인
display(qc.draw("mpl"))
print("Abstract depth:", qc.depth())
assert qc.depth() == 4, f"Expected depth 4, got {qc.depth()}"

# 라인 부분그래프 0-1-2-3-4에 맞춰 트랜스파일
qc_m = qc.copy()
qc_m.measure_all()
pm = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=[0, 1, 2, 3, 4]
)
isa_qc = pm.run(qc_m)
print(f"Transpiled depth: {isa_qc.depth()}")
print(f"CZ gates: {isa_qc.count_ops().get('cz', 0)}")

In [ ]:
grade_lab1_ex5(qc)

### 하드웨어 토폴로지: 헤비-헥스

FakeTorino는 **헤비-헥스**(heavy-hex) 연결을 가진 **Heron 프로세서**를 모델링합니다. 큐비트 133개가 독특한 패턴으로 놓입니다.

- **수평 체인**의 큐비트가 각 행을 가로지릅니다
- **수직 브리지 큐비트**가 엇갈린 자리에서 행들을 잇습니다
- 큐비트 대부분은 **차수 2**입니다(이웃 둘)
- **분기점 큐비트**는 **차수 3**입니다(이웃 셋). 길이 갈라지는 지점이죠

이 토폴로지는 완전 연결(all-to-all)이 *아닙니다*. 이웃이 아닌 큐비트 사이의 2큐비트 게이트는 SWAP 게이트를 부르고, SWAP은 깊이를 크게 늘립니다. 헤비-헥스에서 효율적인 회로를 짠다는 건 연결을 거스르지 않고 그 결을 따라간다는 뜻입니다.

FakeTorino 커플링 맵의 일부를 그려 구조를 봅시다.

In [ ]:
# FakeTorino 일부의 헤비-헥스 연결성 시각화
import networkx as nx

cm = backend.coupling_map
# 연습 6의 T자 접합을 보여줄 만큼 큐비트를 포함합니다: 큐비트 25의 세 번째
# 이웃은 큐비트 35이고(35-44가 수직 다리를 이룸), 이는 처음 두 행 너머에
# 있으므로 33에서 멈추지 않고 큐비트 0-45를 그립니다.
sub_qubits = set(range(46))

G = nx.Graph()
for q in sub_qubits:
    G.add_node(q)
for q in sub_qubits:
    for nb in cm.neighbors(q):
        if nb in sub_qubits:
            G.add_edge(q, nb)

# 차수 3 분기점 찾기
junctions = [q for q in sub_qubits if G.degree(q) == 3]
colors = ["#ff6b6b" if q in junctions else "#4ecdc4" for q in G.nodes()]

fig, ax = plt.subplots(figsize=(15, 6))
pos = nx.kamada_kawai_layout(G)
nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=400,
    font_size=9,
    node_color=colors,
    edge_color="#888",
    width=2,
    ax=ax,
)
ax.set_title("FakeTorino: Qubits 0-45 (red = degree-3 junctions)")
plt.tight_layout()
display(fig)
plt.close(fig)

print("Degree-3 junctions (qubits 0-45):", sorted(junctions))

### 연습 6: 헤비-헥스 부분그래프 위의 7큐비트 GHZ

FakeTorino의 이 T자 부분그래프를 봅시다.

```
23 -- 24 -- 25 -- 26 -- 27
            |
           35
            |
           44
```

큐비트 25는 **차수 3 분기점**으로, 큐비트 24, 26, 35에 이어집니다. 위에 그린 헤비-헥스 토폴로지 그래프에서 이 T자 모양(큐비트 25와 그 세 이웃, 그리고 25–35–44 다리)을 그대로 찾을 수 있습니다.

**과제:** 이 부분그래프 위에 **가능한 한 얕은 깊이**로 7큐비트 GHZ 회로를 만드세요.

<details>
<summary><b>힌트</b></summary>

- 어느 큐비트에서 시작할까요? (분기점이 어디 있는지 생각해 보세요.)
- 한 큐비트는 층마다 CX 게이트 하나에만 낄 수 있다는 걸 기억하세요.
- 추상 큐비트를 이렇게 대응시키세요: 0→q23, 1→q24, 2→q25, 3→q26, 4→q27, 5→q35, 6→q44.
</details>

In [ ]:
# 부분그래프 인접 관계 (추상 큐비트 번호):
# 0(q23) -- 1(q24) -- 2(q25) -- 3(q26) -- 4(q27)
#                      |
#                    5(q35)
#                      |
#                    6(q44)

qc = QuantumCircuit(7)

## 여기에 코드를 작성하세요: 가장 좋은 큐비트에서 시작하는 GHZ를 만드세요 ##


# 확인
display(qc.draw("mpl"))
print("Abstract depth:", qc.depth())

# 물리 배치에 맞춰 트랜스파일
qc_m = qc.copy()
qc_m.measure_all()
pm = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=[23, 24, 25, 26, 27, 35, 44]
)
isa_qc = pm.run(qc_m)
print(f"Transpiled depth: {isa_qc.depth()}")
print(f"CZ gates: {isa_qc.count_ops().get('cz', 0)}")

In [ ]:
grade_lab1_ex6(qc)

### 연습 7: 마지막 도전 — 64큐비트 GHZ

이제 배운 걸 다 모아 봅시다. **64큐비트**(FakeTorino의 큐비트 0–63)에 대한 효율적인 GHZ 상태를 만들고 트랜스파일하는 게 이번 과제입니다.

<details>
<summary><b>처음이신가요? "BFS 스패닝 트리"가 뭐고 왜 쓰나요?</b></summary>

**스패닝 트리**는 부분그래프의 모든 큐비트를 고리 없이 잇는 간선들의 모음입니다. **너비 우선 탐색(BFS)**은 고른 *중심* 큐비트에서 시작해 바깥으로 고리를 넓혀 가며 트리를 만듭니다. 먼저 그 이웃 전부(거리 1), 다음으로 그 이웃들의 미방문 이웃(거리 2), 이런 식이죠.

이게 회로 깊이에 왜 중요할까요? 큐비트마다 그 BFS *부모*와 얽히면, GHZ 회로의 깊이는 결국 트리에서 **뿌리부터 잎까지의 가장 긴 거리**가 됩니다. 잘 가운데 잡힌 큐비트에서 시작하면 그 거리를, 따라서 깊이를 하드웨어 그래프가 허락하는 만큼 짧게 유지합니다. (Part 2의 "가운데에서 시작하기"를 라인에서 임의의 그래프로 넓힌 셈입니다.)

배경: [breadth-first search](https://en.wikipedia.org/wiki/Breadth-first_search) · [`CouplingMap`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap).
</details>

**전략:**
1. 백엔드의 커플링 맵으로 큐비트 0–63의 연결을 파악합니다
2. 잘 고른 중심 큐비트에서 BFS(너비 우선 탐색)로 **스패닝 트리**를 만듭니다
3. 트리를 따라 층별로 얽혀 GHZ 회로를 구성합니다
4. 트랜스파일하고 단순한 선형 GHZ와 깊이를 비교합니다

**도움이 되는 정보:**
- 0–63 부분그래프에서 가장 좋은 중심은 **큐비트 25**입니다(다른 어떤 큐비트까지의 최대 거리가 가장 짧습니다)
- `backend.coupling_map.neighbors(q)`가 큐비트 $q$의 이웃을 줍니다
- BFS 트리는 다음 셀에 만들어 뒀습니다. 여러분이 할 일은 큐비트마다 그 부모와 얽히는 CX를 더하는 한 줄입니다.

In [ ]:
# 도우미: 64큐비트 부분그래프의 차수 3 분기점 보기
sub = set(range(64))
print("Degree-3 junctions in qubits 0-63:")
for q in sorted(sub):
    nbs = [nb for nb in cm.neighbors(q) if nb in sub]
    if len(nbs) >= 3:
        print(f"  Qubit {q}: neighbors {sorted(nbs)}")

In [ ]:
from collections import deque

n = 64
sub = set(range(n))

# 1단계: 중심 큐비트에서 BFS로 스패닝 트리 만들기
center = 25  # 가장 좋은 중심 -- 바꿔 보며 깊이가 어떻게 변하는지 확인하세요!

parent = {center: None}
bfs_depth = {center: 0}
queue = deque([center])

while queue:
    node = queue.popleft()
    for nb in sorted(cm.neighbors(node)):
        if nb in sub and nb not in parent:
            parent[nb] = node
            bfs_depth[nb] = bfs_depth[node] + 1
            queue.append(nb)

print(f"BFS from qubit {center}: reached {len(parent)} qubits")
print(f"Max BFS depth: {max(bfs_depth.values())}")

# 2단계: BFS 트리를 따라 층별로 GHZ 회로 만들기
qc = QuantumCircuit(n)
qc.h(center)

max_d = max(bfs_depth.values())
for d in range(1, max_d + 1):
    for q in range(n):
        if q in bfs_depth and bfs_depth[q] == d:
            # ===>  여기에 코드 한 줄을 작성하세요 <===
            # 단일 CX로 큐비트 q를 그 BFS 부모 parent[q]와 얽히게 하세요.
            # (아래 `pass`를 그 게이트로 바꾸세요.)
            ##  여기에 코드를 작성하세요: parent[q]에서 q로 CX를 더하세요 ##
            pass

qc.measure_all()
print(f"\nYour GHZ abstract depth: {qc.depth() - 1}")  # -1은 측정 층

# 3단계: 트랜스파일하고 비교
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_qc = pm.run(qc)
print(f"Your GHZ transpiled depth: {isa_qc.depth()}")
print(f"Your GHZ CZ gates: {isa_qc.count_ops().get('cz', 0)}")

# 단순한 방식과 비교
qc_naive = ghz_chain(n)
qc_naive.measure_all()
isa_naive = pm.run(qc_naive)
print(f"\nNaive chain transpiled depth: {isa_naive.depth()}")
print(f"Naive chain CZ gates: {isa_naive.count_ops().get('cz', 0)}")

In [ ]:
grade_lab1_ex7(qc)

### Part 3 요약

**이번에 익힌 것:**
- **트랜스파일**은 추상 회로를 하드웨어 네이티브 회로로 바꿈 (게이트 분해 + 큐비트 경로 배정 + 배치)
- Heron 프로세서는 $\{R_Z, \sqrt{X}, X, CZ\}$를 씀 (H나 CX 없음)
- **헤비-헥스 토폴로지**: 차수 2 체인 큐비트와 차수 3 분기점 큐비트
- SWAP 게이트(이웃 아닌 큐비트에 필요)는 하나당 CZ 3개 비용이니, 토폴로지를 염두에 둔 설계로 피하기
- **BFS 스패닝 트리**: 임의의 하드웨어 그래프에서 효율적인 GHZ 회로를 짜는 체계적인 방법

---

## Part 4: 노이즈 시뮬레이터에서 실행 <a id="part4"></a>

이제 회로 깊이가 노이즈 있는 프로세서에서 결과에 실제로 어떻게 영향을 주는지 봅시다. 실제 Torino 칩의 오류율과 연결을 재현하는 *로컬* 노이즈 시뮬레이터 **FakeTorino**로, 효율적인 GHZ 회로와 단순한 회로를 견줍니다. 이 절은 전부 **로컬에서** 돌아가므로 IBM Quantum 런타임 시간을 **전혀** 쓰지 않습니다. 마지막 코드 셀에서 실제 백엔드로 바꾸는 법을 보여줍니다.

이건 **Qiskit 패턴** 흐름을 따릅니다.
1. **맵**: 문제에 맞는 추상 회로 설계
2. **최적화**: 대상 백엔드에 맞춰 트랜스파일
3. **실행**: `SamplerV2`로 실행
4. **후처리**: 측정 결과 분석

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2

# FakeTorino의 노이즈 모델과 연결성으로 노이즈 시뮬레이터 만들기
aer_backend = AerSimulator.from_backend(FakeTorino())
sampler = SamplerV2(aer_backend)
print(
    f"Ready to run on: {aer_backend.name} ({aer_backend.num_qubits} qubits, noisy simulation)"
)

# 이 로컬 노이즈 시뮬레이터 대신 실제 하드웨어에서 돌리려면 실제 백엔드로 바꾸면 됩니다
# (이건 IBM Quantum 런타임 시간을 소모하며 저장된 계정이 필요합니다):
#
# from qiskit_ibm_runtime import QiskitRuntimeService
# service = QiskitRuntimeService()
# real_backend = service.least_busy(operational=True, simulator=False)
# sampler = SamplerV2(real_backend)

### 실험

15큐비트 GHZ 회로 두 개를 비교합니다(FakeTorino의 0행, 큐비트 0–14 사용).

1. **단순한 방식(팬아웃)**: 큐비트 0에 H, 그다음 CX(0,1), CX(0,2), …, CX(0,14). 하드웨어 연결과 *안 맞습니다*. 큐비트 0은 다른 큐비트 대부분과 안 이어져 있어, 트랜스파일러가 SWAP 게이트를 잔뜩 넣어야 합니다.

2. **효율적인 방식(토폴로지 인식)**: 가운데(큐비트 7)에서 시작해 물리 체인을 따라 양쪽으로 팬아웃. 하드웨어와 *맞으므로* SWAP이 필요 없습니다.

최적화 이론이 맞다면, 효율적인 회로가 충실도가 훨씬 높게 나와야 합니다.

In [ ]:
n = 15

# 단순한 방식: 큐비트 0에서 팬아웃 (하드웨어엔 안 맞음!)
qc_naive = QuantumCircuit(n)
qc_naive.h(0)
for i in range(1, n):
    qc_naive.cx(0, i)
qc_naive.measure_all()

# 효율적인 방식: 물리 체인을 따라 가운데에서 팬
qc_eff = QuantumCircuit(n)
mid = n // 2  # 큐비트 7
qc_eff.h(mid)
for step in range(1, n):
    left = mid - step
    right = mid + step
    if left >= 0:
        qc_eff.cx(left + 1, left)
    if right < n:
        qc_eff.cx(right - 1, right)
    if left <= 0 and right >= n - 1:
        break
qc_eff.measure_all()

# 네이티브 게이트로 완전 트랜스파일 (실행용)
pm = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=list(range(n))
)
isa_naive = pm.run(qc_naive)
isa_eff = pm.run(qc_eff)

# SWAP을 보이게 유지하는 병렬 트랜스파일 (진단용 계수만)
isa_naive_swaps = transpile(
    qc_naive,
    coupling_map=backend.coupling_map,
    basis_gates=["h", "cx", "swap"],
    initial_layout=list(range(n)),
    optimization_level=1,
)
isa_eff_swaps = transpile(
    qc_eff,
    coupling_map=backend.coupling_map,
    basis_gates=["h", "cx", "swap"],
    initial_layout=list(range(n)),
    optimization_level=1,
)
naive_swaps = isa_naive_swaps.count_ops().get("swap", 0)
eff_swaps = isa_eff_swaps.count_ops().get("swap", 0)

print("=== Circuit Comparison ===")
print(
    f"Naive:     SWAPs inserted = {naive_swaps:>2},  "
    f"transpiled depth = {isa_naive.depth():>4},  "
    f"CZ gates = {isa_naive.count_ops().get('cz', 0)}"
)
print(
    f"Efficient: SWAPs inserted = {eff_swaps:>2},  "
    f"transpiled depth = {isa_eff.depth():>4},  "
    f"CZ gates = {isa_eff.count_ops().get('cz', 0)}"
)
print("\n(Each SWAP becomes ~3 CZ gates, which is why the naive CZ count grows so quickly.)")

In [ ]:
# 노이즈 시뮬레이터에서 두 회로 실행.
# SHOTS는 몇 번 샘플링할지 정합니다. 8192가 이상적이지만 무료 Colab/qBraid
# 세션에서는 타임아웃될 수 있으니, 필요하면 낮추세요(여기서는 1024-2048이면 충분).
SHOTS = 2048

job = sampler.run([isa_naive, isa_eff], shots=SHOTS)
result = job.result()

counts_naive = result[0].data.meas.get_counts()
counts_eff = result[1].data.meas.get_counts()
print("Execution complete!")
print(f"Naive:     {sum(counts_naive.values())} shots")
print(f"Efficient: {sum(counts_eff.values())} shots")

In [ ]:
# GHZ 충실도 계산: |00...0⟩ 또는 |11...1⟩에 든 샷의 비율
def ghz_fidelity(counts, n):
    total = sum(counts.values())
    correct = counts.get("0" * n, 0) + counts.get("1" * n, 0)
    return correct / total


fid_naive = ghz_fidelity(counts_naive, n)
fid_eff = ghz_fidelity(counts_eff, n)

print(f"Naive GHZ fidelity:     {fid_naive:.4f}")
print(f"Efficient GHZ fidelity: {fid_eff:.4f}")
if fid_eff > fid_naive:
    improvement = (fid_eff - fid_naive) / fid_naive * 100
    print(f"Improvement: {improvement:.1f}%")

In [ ]:
# 시각화: 측정 결과 나란히 히스토그램
display(
    plot_histogram(
        [counts_naive, counts_eff],
        legend=["Naive (fan-out)", "Efficient (topology-aware)"],
        number_to_keep=8,
        sort="desc",
        title=f"{n}-Qubit GHZ: Naive vs Efficient on FakeTorino",
        figsize=(14, 5),
    )
)

In [ ]:
# 요약 막대그래프: 충실도 vs 트랜스파일된 깊이
fig, ax = plt.subplots(figsize=(8, 5))
labels = ["Naive\n(fan-out)", "Efficient\n(topology-aware)"]
fidelities = [fid_naive, fid_eff]
depths = [isa_naive.depth(), isa_eff.depth()]
colors = ["#ff6b6b", "#4ecdc4"]

bars = ax.bar(
    range(len(labels)), fidelities, color=colors, width=0.5, edgecolor="white"
)
ax.set_ylabel("GHZ Fidelity", fontsize=13)
ax.set_title(f"{n}-Qubit GHZ on FakeTorino: Depth vs Fidelity", fontsize=14)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=12)

# 막대에 트랜스파일된 깊이 표시
for bar, d, f in zip(bars, depths, fidelities):
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 0.02,
        f"depth = {d}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

ax.set_ylim(0, 1.0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
display(fig)
plt.close(fig)

### 무엇이 보이나요?

효율적인 회로는 트랜스파일된 깊이가 더 얕고 2큐비트 게이트가 더 적어서, 노이즈 시뮬레이터에서 충실도가 훨씬 높게 나옵니다. 패턴은 분명합니다.

- **CZ 게이트가 적음** → 2큐비트 오류의 원천이 적음
- **깊이가 얕음** → 큐비트가 결어긋남(decoherence)을 겪을 시간이 적음
- **SWAP 게이트가 없음** → 경로 배정에서 오는 군더더기가 없음

> **절대 수치에 놀라지 마세요.** 노이즈 있는 하드웨어에서 15큐비트 GHZ 상태는 정말 어렵습니다. 그래서 여기 원시 충실도는 흔히 **0.4–0.6** 범위입니다. 교과서에서 볼 법한 ~0.99가 아니죠. 2큐비트 게이트마다 오류가 조금씩 더해지는데, 그런 게이트가 많으니까요. 이 실험에서 중요한 건 효율적인 회로가 단순한 회로보다 얼마나 나아지는지, 그 **상대적** 차이입니다. 바로 그 격차가 핵심입니다.

실제 하드웨어에서는 차이가 훨씬 극적입니다. **회로 최적화가 탁상공론이 아니라는** 걸 보여주죠. 여러분이 하는 양자 계산의 품질에 곧바로 영향을 줍니다.

### 앞으로: Lab 2

Lab 2에서는 새 IBM **Nighthawk** 프로세서를 **Heron** 프로세서(이 랩에서 다뤄 온 구조)와 비교합니다. 둘은 연결 패턴이 다릅니다. Heron은 헤비-헥스를 쓰고, Nighthawk는 큐비트당 이웃이 더 많은 촘촘한 그래프를 씁니다. 이 구조 차이가 회로 성능에 곧바로 영향을 줍니다.

여기서 쌓은 기술(효율적인 GHZ 상태 만들기, 깊이 이해하기, 헤비-헥스 토폴로지 다루기)은 Lab 2로 그대로 이어집니다. Lab 2에서는 GHZ 회로로 두 프로세서를 벤치마크하고 비교합니다.

Lab 2에서 만나요! 🚀

---

## Lab 1 완료: 최종 요약

**배운 것:**

| 개념 | 핵심 |
|---------|-------------|
| **X, H, CX 게이트** | 양자 회로의 구성 요소 |
| **중첩** | H 게이트가 $\frac{1}{\sqrt{2}}(\lvert 0\rangle + \lvert 1\rangle)$을 만듦 |
| **얽힘** | CX + 중첩이 맞물린 큐비트 쌍을 만듦 |
| **벨 & GHZ 상태** | 큐비트 2개와 N개의 얽힘 |
| **회로 깊이** | 순차적 게이트 층의 개수, 얕을수록 노이즈 적음 |
| **재귀적 팬아웃** | $O(N)$ 대신 $O(\log N)$ 깊이의 GHZ |
| **트랜스파일** | 추상 → 하드웨어 네이티브 회로 |
| **헤비-헥스 토폴로지** | 차수 2 체인 + 차수 3 분기점 |
| **토폴로지 인식 설계** | 하드웨어 연결을 따라가 SWAP 피하기 |
| **노이즈 실행** | 깊이 얕음 → 실제 장치에서 충실도 높음 |

# 추가 정보

**작성자:** James Weaver

**한글화:** Soyoung Shin

**버전:** 1.0.0